# 03. Variable Window: Longest Valid Window

Window size changes. We want the **longest** window that satisfies a condition.

**Learning order:** IDEA → TEMPLATE → NO REPEATS → AT MOST K DISTINCT → CHARACTER REPLACEMENT → AT MOST K ZEROES → INTERVIEW THINKING

## 1. Idea (expand and shrink)

- `right` moves forward every step: **expand**.
- If the window becomes **invalid**, move `left` forward until it is valid again: **shrink**.
- After the window is valid, record its length `right - left + 1`.

Real life: "longest stretch of days where I ate at most 2 different types of fruit". Add a new day. If you now have 3 types, drop days from the start until you are back to 2.

**Why `left` never goes back:** once a part is invalid together with something later, it will not become valid again by adding more elements. So we only move forward → O(n).

## 2. Template (longest)

```python
left = 0
for right in range(len(arr)):
    add(arr[right])
    while window_is_invalid():
        remove(arr[left])
        left += 1
    best = max(best, right - left + 1)
```

Use **`while`**, not `if`: one new element can force several removals.

## 3. Longest Substring Without Repeating Characters

**Valid:** all characters in the window are unique.
Keep a set. If `s[right]` is already in the set, remove from the left until it is gone.

In [1]:
def length_of_longest_substring(s):
    seen = set()
    left = 0
    best = 0
    for right in range(len(s)):
        while s[right] in seen:         # invalid: duplicate
            seen.remove(s[left])
            left += 1
        seen.add(s[right])
        best = max(best, right - left + 1)
    return best

print(length_of_longest_substring("abcabcbb"))   # 3
print(length_of_longest_substring("bbbbb"))      # 1
print(length_of_longest_substring("pwwkew"))     # 3

3
1
3


In [2]:
# Trace on "abcabcbb"
s = "abcabcbb"
seen, left = set(), 0
for right in range(len(s)):
    while s[right] in seen:
        seen.remove(s[left])
        left += 1
    seen.add(s[right])
    print(f"right={right} window='{s[left:right + 1]}' len={right - left + 1}")

right=0 window='a' len=1
right=1 window='ab' len=2
right=2 window='abc' len=3
right=3 window='bca' len=3
right=4 window='cab' len=3
right=5 window='abc' len=3
right=6 window='cb' len=2
right=7 window='b' len=1


## 4. Longest with at most K distinct

**Valid:** number of distinct characters ≤ `k`. Keep a **count dictionary**. When a count becomes 0, delete the key so `len(dict)` is the number of distinct items.

This works for strings and lists, so *Fruit Into Baskets* is the same with `k = 2`.

In [3]:
from collections import defaultdict

def longest_at_most_k_distinct(seq, k):
    count = defaultdict(int)
    left = 0
    best = 0
    for right in range(len(seq)):
        count[seq[right]] += 1
        while len(count) > k:
            count[seq[left]] -= 1
            if count[seq[left]] == 0:
                del count[seq[left]]
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_at_most_k_distinct("eceba", 2))          # 3  ("ece")
print(longest_at_most_k_distinct([1, 2, 3, 2, 2], 2))  # 4  (fruit baskets)

3
4


## 5. Longest Repeating Character Replacement

You may replace at most `k` characters. Find the longest substring of one repeated letter.

**Key idea:** in a window, we keep the most frequent letter and replace all others. Replacements needed = `window length - max frequency`.
**Valid:** `window length - max_freq <= k`.

(`max_freq` is never decreased when shrinking. This is fine because the answer only grows when `max_freq` really grows, so a stale value never gives a wrong answer.)

In [4]:
def character_replacement(s, k):
    count = defaultdict(int)
    left = 0
    max_freq = 0
    best = 0
    for right in range(len(s)):
        count[s[right]] += 1
        max_freq = max(max_freq, count[s[right]])
        while (right - left + 1) - max_freq > k:
            count[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(character_replacement("AABABBA", 1))   # 4
print(character_replacement("ABAB", 2))      # 4

4
4


## 6. Max Consecutive Ones III

Flip at most `k` zeros to ones. Longest run of ones. **Valid:** zeros in window ≤ `k`.

In [5]:
def longest_ones(nums, k):
    zeros = 0
    left = 0
    best = 0
    for right in range(len(nums)):
        if nums[right] == 0:
            zeros += 1
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))   # 6

6


## 7. Interview thinking

**Signals:** "longest substring/subarray with ...", "at most k", "no repeating", "with k replacements/flips".

**Say:** "I expand `right`. When the window breaks the rule, I shrink from `left` until it is valid, and I record the best length after that. Each element is added and removed at most once, so O(n)."

**Common mistakes:** `if` instead of `while`, updating `best` before the window is valid, not deleting zero-count keys, off-by-one in length (`right - left + 1`), forgetting to define precisely what *valid* means.

## Quick Revision

- Longest: expand → while invalid shrink → `best = max(best, right-left+1)`.
- Define **valid** first (no repeats, ≤k distinct, ≤k zeros, `len - max_freq ≤ k`).
- Use `while` for shrinking. `left` never moves backward.
- Count dict: delete key when count hits 0.
- O(n) time, each element added once and removed once.